In [1]:
# Очистка места
!rm -rf /content/vlm_images/*
!rm -rf /content/coco2017/*
!rm -rf ~/.cache/huggingface/datasets/*

import gc
gc.collect()

from google.colab import drive
drive.mount('/content/drive')

WORK_DIR = "/content/drive/MyDrive/VK_VLM_Project"
IMG_DIR = "/content/vlm_images"
!mkdir -p $WORK_DIR $IMG_DIR

import os, torch
from datasets import load_dataset
from PIL import Image

print(f"✅ Свободное место: {os.statvfs('/content').f_bavail * os.statvfs('/content').f_frsize / 1e9:.2f} GB")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Свободное место: 65.32 GB


In [2]:
print("📦 Загрузка датасета: deepvk/LLaVA-Instruct-ru (только 200 примеров)")
train_dataset = load_dataset("deepvk/LLaVA-Instruct-ru", split="train[:200]")

print(f"✅ Загружено: {len(train_dataset)} примеров")
print(f"📋 Колонки: {train_dataset.column_names}")

📦 Загрузка датасета: deepvk/LLaVA-Instruct-ru (только 200 примеров)


Generating train split:   0%|          | 0/109905 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/34075 [00:00<?, ? examples/s]

✅ Загружено: 200 примеров
📋 Колонки: ['conversations', 'type', 'id', 'image']


In [3]:
import os
import requests
from PIL import Image
from io import BytesIO

def format_for_qwen(example, idx):
    """
    Конвертирует пример deepvk/LLaVA-Instruct-ru в формат чата Qwen2-VL
    Загружает картинки напрямую из COCO API
    """
    img_path = f"{IMG_DIR}/img_{idx}.jpg"

    # === ИЗВЛЕЧЕНИЕ КАРТИНКИ ===
    image_path_str = example['image']  # "coco/train2017/000000253464.jpg"

    # Извлекаем ID картинки из пути
    try:
        filename = os.path.basename(image_path_str)  # "000000253464.jpg"
        image_id = int(filename.split('.')[0])  # 253464
    except Exception as e:
        print(f"⚠️ Не удалось извлечь ID из пути: {image_path_str}")
        return None

    # === ЗАГРУЗКА КАРТИНКИ ИЗ COCO НАПРЯМУЮ ===
    # Если картинка ещё не загружена
    if not os.path.exists(img_path):
        # Формируем URL для COCO 2017
        # Формат: http://images.cocodataset.org/train2017/000000253464.jpg
        coco_url = f"http://images.cocodataset.org/train2017/{image_id:012d}.jpg"

        try:
            response = requests.get(coco_url, timeout=10)
            if response.status_code == 200:
                img = Image.open(BytesIO(response.content)).convert("RGB")
                img.save(img_path, quality=85)
            else:
                print(f"⚠️ Не удалось загрузить картинку {image_id} (HTTP {response.status_code})")
                return None
        except Exception as e:
            print(f"⚠️ Ошибка загрузки картинки {image_id}: {e}")
            return None

    # === ИЗВЛЕЧЕНИЕ ВОПРОСА И ОТВЕТА ===
    conversations = example['conversations']

    question = ""
    answer = ""

    for msg in conversations:
        if msg['from'] == 'human':
            question = msg['value']
            # Убираем тег <image>\n из вопроса
            question = question.replace('<image>\n', '').replace('<image>', '').strip()
        elif msg['from'] == 'gpt':
            answer = msg['value']

    if not question or not answer:
        print(f"⚠️ Пример {idx}: не удалось извлечь question/answer")
        return None

    return {
        "messages": [
            {"role": "user", "content": [
                {"type": "image", "image": img_path},
                {"type": "text", "text": question}
            ]},
            {"role": "assistant", "content": [
                {"type": "text", "text": answer}
            ]}
        ]
    }

# Применяем форматирование
print("🔄 Подготовка данных (загрузка картинок из COCO)...")
formatted_dataset = train_dataset.map(
    format_for_qwen,
    with_indices=True,
    remove_columns=train_dataset.column_names
)

# Удаляем None значения
formatted_dataset = formatted_dataset.filter(lambda x: x is not None)

print(f"✅ Подготовлено {len(formatted_dataset)} примеров для обучения")
print(f"\n=== Пример после форматирования ===")
print(formatted_dataset[0])

🔄 Подготовка данных (загрузка картинок из COCO)...


Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Filter:   0%|          | 0/200 [00:00<?, ? examples/s]

✅ Подготовлено 200 примеров для обучения

=== Пример после форматирования ===
{'messages': [{'role': 'user', 'content': [{'type': 'image', 'image': '/content/vlm_images/img_0.jpg'}, {'type': 'text', 'text': 'Что делает обстановку этой ванны такой особенной по сравнению с другими?'}]}, {'role': 'assistant', 'content': [{'type': 'text', 'text': 'Эта ванная комната выделяется своей чистотой и опрятностью. На изображении видно, что ванная идеально чистая, свет включен. Наличие цветов, стоящих на столешнице, добавляет элемент уюта и свежести в интерьер. Деревянные шкафчики придают теплоту и натуральность обстановке. Зелено-белый полосатый душевой занавес создает яркий акцент и придает привлекательность дизайну ванной комнаты.'}]}]}


In [4]:
import transformers
print(f"Версия transformers: {transformers.__version__}")

# Для Qwen2-VL используем специализированный класс
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model
import torch

print("✅ Импорт успешен!")

Версия transformers: 5.12.1


✅ Импорт успешен!


In [5]:
print("🤖 Загрузка модели Qwen2-VL-2B-Instruct...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = Qwen2VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2-VL-2B-Instruct",
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

processor = AutoProcessor.from_pretrained(
    "Qwen/Qwen2-VL-2B-Instruct",
    trust_remote_code=True
)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

print(f"✅ Модель загружена. VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB")

🤖 Загрузка модели Qwen2-VL-2B-Instruct...


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


trainable params: 2,179,072 || all params: 2,211,164,672 || trainable%: 0.0985
✅ Модель загружена. VRAM: 1.53 GB


In [6]:
from trl import SFTConfig, SFTTrainer

print("🚀 Начинаем обучение...")

training_args = SFTConfig(
    output_dir=f"{WORK_DIR}/checkpoints",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    bf16=True,
    logging_steps=20,
    save_strategy="steps",
    save_steps=100,
    max_length=512,
    report_to="none",
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=formatted_dataset,
    processing_class=processor,
)

trainer.train()

trainer.save_model(f"{WORK_DIR}/final_adapter")
processor.save_pretrained(f"{WORK_DIR}/final_adapter")

print(f"✅ Обучение завершено! Веса сохранены в {WORK_DIR}/final_adapter")

del trainer
gc.collect()
torch.cuda.empty_cache()

🚀 Начинаем обучение...


Tokenizing train dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 151645, 'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
20,6.622356
40,6.168330


✅ Обучение завершено! Веса сохранены в /content/drive/MyDrive/VK_VLM_Project/final_adapter


In [7]:
from peft import PeftModel
from qwen_vl_utils import process_vision_info

print("🧪 Тестирование модели...")

base_model = Qwen2VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2-VL-2B-Instruct",
    device_map="auto",
    trust_remote_code=True
)
model = PeftModel.from_pretrained(base_model, f"{WORK_DIR}/final_adapter")
processor = AutoProcessor.from_pretrained(f"{WORK_DIR}/final_adapter", trust_remote_code=True)

test_messages = [
    {"role": "user", "content": [
        {"type": "image", "image": f"{IMG_DIR}/img_0.jpg"},
        {"type": "text", "text": "Что изображено на картинке?"}
    ]}
]

text = processor.apply_chat_template(test_messages, tokenize=False, add_generation_prompt=True)
image_inputs, _ = process_vision_info(test_messages)
inputs = processor(
    text=[text],
    images=image_inputs,
    padding=True,
    return_tensors="pt"
).to("cuda")

output_ids = model.generate(**inputs, max_new_tokens=128)
output = processor.batch_decode(
    output_ids[:, inputs.input_ids.shape[1]:],
    skip_special_tokens=True
)
print(f"\n💬 Ответ модели:\n{output[0]}")

🧪 Тестирование модели...


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]


💬 Ответ модели:
На картинке изображена ванная комната с зеленой полосатой душевой кабинкой, белым туалетом, зеленым полотенцем, вазой с желтыми цветами, зеркалом с зелеными шторками, зеркальной подставкой, зеркальной подставкой для туалетной бумаги, зеркальной подставкой для мыла, зеркальной подставкой для мыла, зеркальной подставкой для мы


In [12]:
from datasets import load_dataset, get_dataset_config_names, get_dataset_split_names

print("🔍 Проверяем структуру бенчмарков...\n")

# === GQA-ru ===
print("=== GQA-ru ===")
gqa_configs = get_dataset_config_names("deepvk/GQA-ru")
print(f"Конфигурации: {gqa_configs}")

for config in gqa_configs:
    splits = get_dataset_split_names("deepvk/GQA-ru", config)
    print(f"  {config} -> сплиты: {splits}")

# === MMBench-ru ===
print("\n=== MMBench-ru ===")
mmbench_configs = get_dataset_config_names("deepvk/MMBench-ru")
print(f"Конфигурации: {mmbench_configs}")

for config in mmbench_configs:
    splits = get_dataset_split_names("deepvk/MMBench-ru", config)
    print(f"  {config} -> сплиты: {splits}")

🔍 Проверяем структуру бенчмарков...

=== GQA-ru ===
Конфигурации: ['testdev_balanced_images', 'testdev_balanced_instructions', 'train_balanced_images', 'train_balanced_instructions']
  testdev_balanced_images -> сплиты: ['testdev']
  testdev_balanced_instructions -> сплиты: ['testdev']
  train_balanced_images -> сплиты: ['train']
  train_balanced_instructions -> сплиты: ['train']

=== MMBench-ru ===
Конфигурации: ['default']
  default -> сплиты: ['dev']


In [13]:
from datasets import load_dataset

print("📊 Загрузка бенчмарков...\n")

# GQA-ru: конфигурация testdev_balanced_images, сплит testdev
gqa_dataset = load_dataset(
    "deepvk/GQA-ru",
    "testdev_balanced_images",
    split="testdev[:50]"  # ИСПРАВЛЕНО: было "test[:50]"
)
print(f"✅ GQA-ru: {len(gqa_dataset)} примеров")
print(f"   Колонки: {gqa_dataset.column_names}")
print(f"   Пример: {gqa_dataset[0]}")

# MMBench-ru: пробуем разные варианты
print()
try:
    # Пробуем самый частый вариант
    mmbench_dataset = load_dataset("deepvk/MMBench-ru", "dev", split="dev[:50]")
    print(f"✅ MMBench-ru (dev): {len(mmbench_dataset)} примеров")
except Exception as e:
    print(f"⚠️ Конфигурация 'dev' не работает: {e}")
    # Пробуем альтернативу
    try:
        mmbench_dataset = load_dataset("deepvk/MMBench-ru", split="test[:50]")
        print(f"✅ MMBench-ru (без конфигурации): {len(mmbench_dataset)} примеров")
    except Exception as e2:
        print(f"⚠️ Ошибка: {e2}")
        mmbench_dataset = None

if mmbench_dataset:
    print(f"   Колонки: {mmbench_dataset.column_names}")
    print(f"   Пример: {mmbench_dataset[0]}")

📊 Загрузка бенчмарков...

✅ GQA-ru: 50 примеров
   Колонки: ['id', 'image']
   Пример: {'id': 'n161313', 'image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=500x280 at 0x7FF4F7ED5C40>}

⚠️ Конфигурация 'dev' не работает: BuilderConfig 'dev' not found. Available: ['default']


mmbench_ru_dev.parquet:   0%|          | 0.00/92.7M [00:00<?, ?B/s]

Generating dev split:   0%|          | 0/3910 [00:00<?, ? examples/s]

⚠️ Ошибка: Unknown split "test". Should be one of ['dev'].


In [17]:
import json
import os
from PIL import Image
from datasets import load_dataset

print("📊 Полная оценка на GQA-ru и MMBench-ru...\n")

# === ЗАГРУЗКА GQA-RU ===
print("📦 Загрузка GQA-ru...")

# Загружаем картинки
gqa_images = load_dataset("deepvk/GQA-ru", "testdev_balanced_images", split="testdev")
print(f"  Картинки: {len(gqa_images)} примеров")

# Загружаем вопросы и ответы
gqa_instructions = load_dataset("deepvk/GQA-ru", "testdev_balanced_instructions", split="testdev[:50]")
print(f"  Вопросы: {len(gqa_instructions)} примеров")

# Создаём словарь для быстрого поиска картинок по ID
image_dict = {item['id']: item['image'] for item in gqa_images}
print(f"  Создан словарь для {len(image_dict)} картинок\n")

# === ЗАГРУЗКА MMBENCH-RU ===
print("📦 Загрузка MMBench-ru...")
mmbench_dataset = load_dataset("deepvk/MMBench-ru", "default", split="dev[:50]")
print(f"  Загружено: {len(mmbench_dataset)} примеров\n")

# === ФУНКЦИЯ ОЦЕНКИ GQA ===
def evaluate_gqa(gqa_instructions, image_dict, max_samples=50):
    """Оценка на GQA-ru (связываем картинки и вопросы по ID)"""
    correct = 0
    processed = 0
    total = min(max_samples, len(gqa_instructions))

    print(f"🔍 Оценка на GQA-ru ({total} примеров)...\n")

    for i in range(total):
        instruction = gqa_instructions[i]

        # Извлекаем данные
        image_id = instruction.get("imageId")
        question = instruction.get("question", "")
        ground_truth = instruction.get("answer", "")

        if not all([image_id, question, ground_truth]):
            continue

        # Получаем картинку из словаря
        image = image_dict.get(image_id)
        if not image:
            continue

        # Сохраняем картинку
        img_path = f"{IMG_DIR}/eval_gqa_{i}.jpg"
        if isinstance(image, Image.Image):
            image.convert("RGB").save(img_path)
        else:
            continue

        # Инференс
        messages = [
            {"role": "user", "content": [
                {"type": "image", "image": img_path},
                {"type": "text", "text": question}
            ]}
        ]

        try:
            text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            image_inputs, _ = process_vision_info(messages)
            inputs = processor(
                text=[text],
                images=image_inputs,
                padding=True,
                return_tensors="pt"
            ).to("cuda")

            output_ids = model.generate(**inputs, max_new_tokens=64)
            prediction = processor.batch_decode(
                output_ids[:, inputs.input_ids.shape[1]:],
                skip_special_tokens=True
            )[0].strip()

            # Проверяем правильность (для GQA: точное совпадение или вхождение)
            gt_lower = ground_truth.strip().lower()
            pred_lower = prediction.strip().lower()

            is_correct = (gt_lower in pred_lower) or (pred_lower in gt_lower)
            if is_correct:
                correct += 1

            processed += 1

            # Выводим примеры
            if processed <= 3 or processed % 10 == 0:
                print(f"Пример {processed}:")
                print(f"  Вопрос: {question}")
                print(f"  Правильный ответ: {ground_truth}")
                print(f"  Предсказание: {prediction[:100]}")
                print(f"  {'✓ Верно' if is_correct else '✗ Неверно'}\n")

            if processed % 10 == 0:
                print(f"  Прогресс: {processed}/{total}, точность: {correct/processed:.2%}\n")

        except Exception as e:
            continue

    accuracy = correct / processed if processed > 0 else 0
    print(f"✅ GQA-ru Accuracy: {accuracy:.2%} ({correct}/{processed})\n")
    return accuracy

# === ФУНКЦИЯ ОЦЕНКИ MMBENCH ===
def evaluate_mmbench(dataset, max_samples=50):
    """Оценка на MMBench-ru (выбор правильного варианта A/B/C/D)"""
    correct = 0
    processed = 0
    total = min(max_samples, len(dataset))

    print(f"🔍 Оценка на MMBench-ru ({total} примеров)...\n")

    for i in range(total):
        example = dataset[i]

        # Извлекаем данные
        image = example.get("image")
        question = example.get("question", "")
        hint = example.get("hint", "")

        # Варианты ответов
        options = {}
        for opt in ["A", "B", "C", "D"]:
            if opt in example and example[opt] and str(example[opt]).lower() != "nan":
                options[opt] = example[opt]

        # Правильный ответ
        correct_answer = example.get("answer", "")

        if not all([image, question, correct_answer]):
            continue

        # Формируем полный вопрос
        full_question = question
        if hint and str(hint).lower() != "nan":
            full_question = f"{hint}\n\n{question}"

        if options:
            full_question += "\n\nВарианты ответов:"
            for opt, text in options.items():
                full_question += f"\n{opt}. {text}"
            full_question += "\n\nВыберите правильный вариант (A/B/C/D)."

        # Сохраняем картинку
        img_path = f"{IMG_DIR}/eval_mmbench_{i}.jpg"
        if isinstance(image, Image.Image):
            image.convert("RGB").save(img_path)
        else:
            continue

        # Инференс
        messages = [
            {"role": "user", "content": [
                {"type": "image", "image": img_path},
                {"type": "text", "text": full_question}
            ]}
        ]

        try:
            text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            image_inputs, _ = process_vision_info(messages)
            inputs = processor(
                text=[text],
                images=image_inputs,
                padding=True,
                return_tensors="pt"
            ).to("cuda")

            output_ids = model.generate(**inputs, max_new_tokens=128)
            prediction = processor.batch_decode(
                output_ids[:, inputs.input_ids.shape[1]:],
                skip_special_tokens=True
            )[0].strip()

            # Проверяем правильность
            pred_upper = prediction.upper()
            is_correct = False

            if pred_upper.startswith(correct_answer.upper()):
                is_correct = True
            elif correct_answer.upper() in pred_upper[:50]:
                is_correct = True

            if is_correct:
                correct += 1

            processed += 1

            # Выводим примеры
            if processed <= 3 or processed % 10 == 0:
                print(f"Пример {processed}:")
                print(f"  Вопрос: {question[:80]}...")
                print(f"  Правильный ответ: {correct_answer}")
                print(f"  Предсказание: {prediction[:100]}...")
                print(f"  {'✓ Верно' if is_correct else '✗ Неверно'}\n")

            if processed % 10 == 0:
                print(f"  Прогресс: {processed}/{total}, точность: {correct/processed:.2%}\n")

        except Exception as e:
            continue

    accuracy = correct / processed if processed > 0 else 0
    print(f"✅ MMBench-ru Accuracy: {accuracy:.2%} ({correct}/{processed})\n")
    return accuracy

# === ЗАПУСК ОЦЕНКИ ===
gqa_accuracy = evaluate_gqa(gqa_instructions, image_dict, max_samples=50)
mmbench_accuracy = evaluate_mmbench(mmbench_dataset, max_samples=50)

# === СОХРАНЕНИЕ РЕЗУЛЬТАТОВ ===
results = {
    "model": "Qwen2-VL-2B + LoRA on deepvk/LLaVA-Instruct-ru",
    "training_samples": len(formatted_dataset) if 'formatted_dataset' in locals() else 200,
    "GQA-ru_accuracy": gqa_accuracy,
    "MMBench-ru_accuracy": mmbench_accuracy,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
    "notes": "GQA-ru: картинки и вопросы загружены из разных конфигураций и связаны по ID."
}

with open(f"{WORK_DIR}/metrics_report.json", "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f"📄 Результаты сохранены в {WORK_DIR}/metrics_report.json")
print(json.dumps(results, ensure_ascii=False, indent=2))

📊 Полная оценка на GQA-ru и MMBench-ru...

📦 Загрузка GQA-ru...
  Картинки: 398 примеров
  Вопросы: 50 примеров
  Создан словарь для 398 картинок

📦 Загрузка MMBench-ru...
  Загружено: 50 примеров

🔍 Оценка на GQA-ru (50 примеров)...

Пример 1:
  Вопрос: Пасмурно?
  Правильный ответ: нет
  Предсказание: Извините, но я не могу предоставить информацию о погоде.
  ✗ Неверно

Пример 2:
  Вопрос: Кто носит платье?
  Правильный ответ: женщины
  Предсказание: На фотографии нет человека, носящего платье.
  ✗ Неверно

Пример 3:
  Вопрос: Посуда на столе выглядит чистой и черной?
  Правильный ответ: нет
  Предсказание: На столе нет чистой и черной посуды.
  ✓ Верно

Пример 10:
  Вопрос: Что находится вокруг открытого окна?
  Правильный ответ: шторы
  Предсказание: Вокруг открытого окна находится кровать, которая покрыта одеялом с изображением цветов. Вокруг крова
  ✗ Неверно

  Прогресс: 10/50, точность: 40.00%

Пример 20:
  Вопрос: Есть ли кровати рядом с маленькой розеткой?
  Правильный ответ: